# 02 — Labeling Emosi dengan SenticNet

Notebook ini mendokumentasikan proses pemberian label emosi pada hasil preprocessing komentar YouTube (`hasil_stemming.csv`) menggunakan leksikon **SenticNet**.

---

## Konteks

Setelah preprocessing di notebook `01_preprocessing_youtube.ipynb`, setiap komentar sudah dalam bentuk token yang telah di-*stem*. Langkah selanjutnya adalah memberikan label emosi secara otomatis menggunakan pendekatan **lexicon-based** dengan SenticNet.

**Prinsip kerja:**
1. Setiap token (dan bigram/trigram) di-*lookup* ke kamus SenticNet
2. Setiap konsep yang ditemukan memiliki `PRIMARY EMOTION` dan `POLARITY INTENSITY`
3. Emosi dominan dalam satu kalimat ditentukan dari emosi yang **paling sering muncul**
4. Komentar yang tidak ada satupun token-nya di SenticNet **dibuang**

> **Catatan Reprodusibilitas:** File `hasil_stemming.csv` (output langsung notebook 01) tidak lagi tersimpan. Notebook ini menggunakan `dataset_emosi_single_label.csv` sebagai representasi output akhir proses ini, dan mendokumentasikan ulang logika yang digunakan.

## 1. Install & Import Library

In [ ]:
!pip install openpyxl sastrawi -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('Import selesai')

## 2. Load Data

In [ ]:
# Load hasil stemming dari notebook 01
# File ini adalah output dari preprocessing_youtube.ipynb
df = pd.read_csv('hasil_stemming.csv')

print(f'Total data: {len(df)} baris')
print(f'Kolom: {df.columns.tolist()}')
df.head()

In [ ]:
# Load SenticNet — leksikon emosi berbahasa Indonesia
senticnet = pd.read_excel('senticnet.xlsx')

print(f'Total entri SenticNet: {len(senticnet)}')
print(f'Kolom: {senticnet.columns.tolist()}')
senticnet.head()

## 3. Eksplorasi SenticNet

In [ ]:
# Distribusi PRIMARY EMOTION di SenticNet
print('Distribusi PRIMARY EMOTION:')
print(senticnet['PRIMARY EMOTION'].value_counts())

## 4. Mapping PRIMARY EMOTION → 7 Label Emosi

SenticNet menggunakan 24 kategori emosi. Kita pemetakan ke **7 emosi dasar** (Plutchik's Wheel of Emotions) yang digunakan dalam proyek ini.

In [ ]:
# Pemetaan 24 emosi SenticNet → 7 emosi target
EMOTION_MAP = {
    # Anger
    '#rage'       : 'Anger',
    '#anger'      : 'Anger',
    '#annoyance'  : 'Anger',
    '#loathing'   : 'Anger',
    # Joy
    '#ecstasy'    : 'Joy',
    '#joy'        : 'Joy',
    '#bliss'      : 'Joy',
    '#serenity'   : 'Joy',
    '#delight'    : 'Joy',
    '#pleasantness': 'Joy',
    '#contentment': 'Joy',
    # Sadness
    '#grief'      : 'Sadness',
    '#sadness'    : 'Sadness',
    '#melancholy' : 'Sadness',
    # Fear
    '#terror'     : 'Fear',
    '#fear'       : 'Fear',
    '#anxiety'    : 'Fear',
    # Anticipation
    '#eagerness'  : 'Anticipation',
    '#enthusiasm' : 'Anticipation',
    '#responsiveness': 'Anticipation',
    # Trust
    '#acceptance' : 'Trust',
    '#calmness'   : 'Trust',
    # Disgust
    '#disgust'    : 'Disgust',
    '#dislike'    : 'Disgust',
}

# Terapkan mapping ke SenticNet
senticnet['emotion_label'] = senticnet['PRIMARY EMOTION'].map(EMOTION_MAP)

# Buat lexicon dict: concept → (emotion_label, polarity_intensity)
lexicon_dict = dict(zip(
    senticnet['CONCEPT'],
    zip(senticnet['emotion_label'], senticnet['POLARITY INTENSITY'])
))

print(f'Total entri lexicon: {len(lexicon_dict)}')
print('\nContoh entri lexicon:')
for k, v in list(lexicon_dict.items())[:5]:
    print(f'  "{k}" → emosi: {v[0]}, skor: {v[1]}')

## 5. Fungsi Labeling

Proses labeling mendukung **bigram** dan **trigram** — dua/tiga kata yang digabung dengan underscore (`_`) dapat dikenali sebagai satu konsep di SenticNet (contoh: `sangat_senang`, `tidak_bisa_tidur`).

In [ ]:
def merge_phrases(tokens, label_mapping):
    """
    Gabungkan token menjadi bigram/trigram jika ada di lexicon.
    Prioritas: trigram > bigram > unigram.
    """
    merged = []
    i = 0
    while i < len(tokens):
        # Coba trigram dulu
        if i + 2 < len(tokens):
            trigram = f"{tokens[i]}_{tokens[i+1]}_{tokens[i+2]}"
            if trigram in label_mapping:
                merged.append(trigram)
                i += 3
                continue
        # Coba bigram
        if i + 1 < len(tokens):
            bigram = f"{tokens[i]}_{tokens[i+1]}"
            if bigram in label_mapping:
                merged.append(bigram)
                i += 2
                continue
        # Unigram
        merged.append(tokens[i])
        i += 1
    return merged


def label_emosi_dominan(text, lexicon):
    """
    Tentukan emosi dominan dari sebuah teks berdasarkan lexicon SenticNet.
    
    Langkah:
    1. Tokenisasi teks
    2. Merge bigram/trigram yang ada di lexicon
    3. Lookup emosi tiap token/phrase
    4. Pilih emosi yang paling sering muncul (majority vote)
    5. Hitung rata-rata skor polaritas absolut
    
    Returns:
        (label_emosi, skor) atau (None, None) jika tidak ada token yang match
    """
    if not isinstance(text, str) or text.strip() == '':
        return None, None
    
    tokens = text.split()
    merged = merge_phrases(tokens, lexicon)
    
    emosi_list = []
    skor_list  = []
    
    for phrase in merged:
        if phrase in lexicon:
            emosi, skor = lexicon[phrase]
            if emosi:  # hanya tambahkan jika ada mapping emosi
                emosi_list.append(emosi)
                skor_list.append(abs(float(skor)))
    
    if not emosi_list:
        return None, None
    
    # Emosi dominan: majority vote
    emosi_dominan = Counter(emosi_list).most_common(1)[0][0]
    
    # Skor: rata-rata absolut polarity intensity
    avg_skor = np.mean(skor_list)
    
    return emosi_dominan, round(avg_skor, 9)


print('Fungsi labeling siap')
print()

# Uji cepat
contoh = 'aku sangat takut dan cemas dengan kondisi ini'
label, skor = label_emosi_dominan(contoh, lexicon_dict)
print(f'Contoh teks : "{contoh}"')
print(f'Label emosi : {label}')
print(f'Skor        : {skor}')

## 6. Terapkan Labeling ke Seluruh Dataset

In [ ]:
print('Memproses labeling... Mohon tunggu.')

# Terapkan ke kolom stemming_text (hasil akhir preprocessing notebook 01)
hasil = df['stemming_text'].apply(lambda x: label_emosi_dominan(x, lexicon_dict))

df['label_single'] = [r[0] for r in hasil]
df['skor']         = [r[1] for r in hasil]

print(f'Selesai!')
print(f'Total input  : {len(df)}')
print(f'Berhasil dilabeli : {df["label_single"].notna().sum()}')
print(f'Tidak terdeteksi  : {df["label_single"].isna().sum()} (tidak ada token di SenticNet)')
df.head()

In [ ]:
# Hapus baris yang tidak terdeteksi emosinya
df_labeled = df.dropna(subset=['label_single']).reset_index(drop=True)

# Simpan hanya kolom yang diperlukan
df_labeled = df_labeled[['normalize_text', 'skor', 'label_single']]

print(f'Dataset setelah filter: {len(df_labeled)} baris')
print()
print('Distribusi label:')
print(df_labeled['label_single'].value_counts())
df_labeled.head()

## 7. Visualisasi Distribusi Label

In [ ]:
label_counts = df_labeled['label_single'].value_counts()
total = len(df_labeled)

plt.figure(figsize=(10, 5))
bars = plt.bar(label_counts.index, label_counts.values,
               color=['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6','#1abc9c','#e67e22'])

for bar, val in zip(bars, label_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val}\n({val/total*100:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.title('Distribusi Label Emosi (Hasil Labeling SenticNet)', fontsize=13, fontweight='bold')
plt.xlabel('Label Emosi')
plt.ylabel('Jumlah Komentar')
plt.tight_layout()
plt.savefig('distribusi_label_emosi.png', dpi=150)
plt.show()

print(f'\n⚠️  Catatan: Terdapat class imbalance — Sadness mendominasi ({label_counts["Sadness"]/total*100:.1f}%)')
print('   Ini akan ditangani pada tahap penggabungan dataset (notebook 03).')

In [ ]:
# Distribusi skor polaritas
plt.figure(figsize=(10, 4))
plt.hist(df_labeled['skor'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
plt.title('Distribusi Skor Polaritas (Rata-rata Abs Polarity Intensity)', fontsize=12)
plt.xlabel('Skor')
plt.ylabel('Frekuensi')
plt.axvline(df_labeled['skor'].mean(), color='red', linestyle='--',
            label=f'Mean: {df_labeled["skor"].mean():.3f}')
plt.legend()
plt.tight_layout()
plt.show()

print(df_labeled['skor'].describe())

## 8. Simpan Output

In [ ]:
df_labeled.to_csv('dataset_emosi_single_label.csv', index=False)

print('File tersimpan: dataset_emosi_single_label.csv')
print(f'Shape         : {df_labeled.shape}')
print(f'Kolom         : {df_labeled.columns.tolist()}')
print()
print('Preview:')
df_labeled.head()

## Ringkasan

| Item | Nilai |
|------|-------|
| Input | `hasil_stemming.csv` (output notebook 01) |
| Leksikon | SenticNet (Bahasa Indonesia) |
| Metode | Lexicon-based majority vote (unigram + bigram + trigram) |
| Emosi mapping | 24 kategori SenticNet → 7 emosi (Plutchik) |
| Output | `dataset_emosi_single_label.csv` |
| Total baris output | ~537 baris |

**Catatan:** Dataset hasil labeling ini memiliki *class imbalance* (Sadness ~67%).
Penanganan imbalance dilakukan pada tahap penggabungan dataset di notebook `03_gabungan_dataset.ipynb`.